# Chicago Crime Analysis — Machine Learning Project

## Contexte Business
La ville de Chicago enregistre **+200 000 crimes par an**. Ce projet répond à deux questions :
1. **Où se concentrent les crimes ?** : K-Means Clustering *(non supervisé)*
2. **Ce crime va-t-il mener à une arrestation ?** : 3 modèles comparés *(supervisé)*

**Valeur opérationnelle :** Aider la police à prioriser ses investigations selon la probabilité de résolution.

| | |
|---|---|
| **Dataset** | Chicago Data Portal — Crimes 2022–2024 |
| **Source** | https://data.cityofchicago.org/resource/ijzp-q8t2.csv |
| **Volume** | ~200 000 lignes |
| **Modèles** | K-Means · Logistic Regression · Random Forest · XGBoost |


## 0 — Imports & Configuration

In [ ]:
# Installations (décommenter si nécessaire sur Colab)
# !pip install pandas numpy matplotlib seaborn scikit-learn xgboost joblib requests --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import requests, warnings, joblib, os
from io import StringIO

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score
)
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', palette='muted', font_scale=1.05)
plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': 'white'})
RANDOM_STATE = 42

print('Imports OK')


## 1 — Chargement des Données

In [ ]:
URL = 'https://data.cityofchicago.org/resource/ijzp-q8t2.csv'
PARAMS = {'$limit': 200000, '$offset': 0, '$order': 'date DESC', '$where': 'year >= 2022'}

print('Chargement depuis l\'API Chicago...')
try:
    r = requests.get(URL, params=PARAMS, timeout=120)
    r.raise_for_status()
    df_raw = pd.read_csv(StringIO(r.text), low_memory=False)
    print(f'{len(df_raw):,} lignes chargées')
except Exception as e:
    print(f'Erreur : {e}')
    df_raw = pd.read_csv(
        'https://data.cityofchicago.org/resource/ijzp-q8t2.csv?$limit=200000&$where=year>=2022',
        low_memory=False
    )

print(f'Shape : {df_raw.shape}')
df_raw.head(3)


## 2 — Exploration des Données (EDA)

In [ ]:
print('='*60)
print('  STATISTIQUES GÉNÉRALES — CHICAGO CRIMES 2022-2024')
print('='*60)
print(f'  Lignes           : {len(df_raw):,}')
print(f'  Types de crimes  : {df_raw["primary_type"].nunique()}')
print(f'  Arrestations     : {df_raw["arrest"].sum():,} ({df_raw["arrest"].mean()*100:.1f}%)')
print(f'  Crimes domestiq. : {df_raw["domestic"].sum():,} ({df_raw["domestic"].mean()*100:.1f}%)')
print()
missing = df_raw.isnull().sum()
print('Valeurs manquantes :')
print(missing[missing > 0].sort_values(ascending=False))


In [ ]:
# ── VIZ 1 : Top 15 types de crimes ──
top15 = df_raw['primary_type'].value_counts().head(15)
fig, ax = plt.subplots(figsize=(12, 6))
colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, 15))
bars = ax.barh(top15.index[::-1], top15.values[::-1], color=colors, edgecolor='white', linewidth=0.4)
ax.set_xlabel('Nombre de crimes', fontsize=12)
ax.set_title('Top 15 Types de Crimes — Chicago 2022–2024', fontsize=14, fontweight='bold')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
for bar, val in zip(bars, top15.values[::-1]):
    ax.text(bar.get_width()+200, bar.get_y()+bar.get_height()/2, f'{val:,}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('viz_01_top15.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── VIZ 2 : Taux d'arrestation par type de crime ──
arrest_by_type = (
    df_raw.groupby('primary_type')['arrest']
    .agg(['mean','count']).query('count > 200')
    .sort_values('mean', ascending=False).head(20)
)
avg = arrest_by_type['mean'].mean()

fig, ax = plt.subplots(figsize=(12, 7))
palette = ['#2ecc71' if v>=0.30 else '#e67e22' if v>=0.15 else '#e74c3c'
           for v in arrest_by_type['mean'].values]
ax.barh(arrest_by_type.index[::-1], arrest_by_type['mean'].values[::-1]*100,
        color=palette[::-1], edgecolor='white', linewidth=0.3)
ax.axvline(x=avg*100, color='#2c3e50', linestyle='--', linewidth=1.8,
           label=f'Moyenne : {avg*100:.1f}%')
ax.set_xlabel("Taux d'arrestation (%)", fontsize=12)
ax.set_title("Taux d'Arrestation par Type de Crime", fontsize=14, fontweight='bold')
patches = [
    mpatches.Patch(color='#2ecc71', label='≥30% (élevé)'),
    mpatches.Patch(color='#e67e22', label='15–30% (moyen)'),
    mpatches.Patch(color='#e74c3c', label='<15% (faible)'),
]
ax.legend(handles=patches, fontsize=10, loc='lower right')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('viz_02_arrest_rate.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── VIZ 3 : Patterns temporels ──
df_raw['date_parsed'] = pd.to_datetime(df_raw['date'], errors='coerce')
df_raw['hour']        = df_raw['date_parsed'].dt.hour
df_raw['day_of_week'] = df_raw['date_parsed'].dt.dayofweek
df_raw['month']       = df_raw['date_parsed'].dt.month

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Heure
hour_arrest = df_raw.groupby(['hour','arrest'])['arrest'].count().unstack(fill_value=0)
hour_arrest.columns = ["Pas d'arrest.","Arrestation"]
hour_arrest.plot(kind='bar', ax=axes[0], color=['#e74c3c','#2ecc71'], width=0.8)
axes[0].set_title('Crimes par Heure', fontweight='bold')
axes[0].set_xlabel('Heure'); axes[0].tick_params(axis='x', rotation=0, labelsize=7)

# Jour
days_fr = ['Lun','Mar','Mer','Jeu','Ven','Sam','Dim']
day_counts = df_raw['day_of_week'].value_counts().sort_index()
arrest_day = df_raw.groupby('day_of_week')['arrest'].mean()
ax2b = axes[1].twinx()
axes[1].bar(range(7), day_counts.values, color='#3498db', alpha=0.5)
ax2b.plot(range(7), arrest_day.values*100, 'o-', color='#e74c3c', linewidth=2)
axes[1].set_xticks(range(7)); axes[1].set_xticklabels(days_fr)
axes[1].set_title('Volume & Taux Arrest. / Jour', fontweight='bold')
ax2b.set_ylabel("Taux arrestation (%)")

# Mois
months_fr = ['Jan','Fév','Mar','Avr','Mai','Jun','Jul','Aoû','Sep','Oct','Nov','Déc']
month_counts = df_raw['month'].value_counts().sort_index()
axes[2].plot(range(1,13), month_counts.values, 'o-', color='#9b59b6', linewidth=2.5, markersize=7)
axes[2].fill_between(range(1,13), month_counts.values, alpha=0.15, color='#9b59b6')
axes[2].set_xticks(range(1,13)); axes[2].set_xticklabels(months_fr, rotation=30)
axes[2].set_title('Saisonnalité des Crimes', fontweight='bold')

plt.suptitle('Analyse Temporelle', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('viz_03_temporal.png', bbox_inches='tight')
plt.show()


## 3 — Nettoyage & Feature Engineering

In [ ]:
df = df_raw.copy()

before = len(df)
df = df.dropna(subset=['latitude','longitude','arrest','primary_type'])
df = df[(df['latitude'] != 0) & (df['longitude'] != 0)]
print(f'Nettoyage : {before:,} -> {len(df):,} lignes ({before-len(df):,} supprimées)')

df['arrest']     = df['arrest'].astype(bool).astype(int)
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
df['domestic']   = df['domestic'].astype(bool).astype(int)

# Encodage du type de crime (label encoding)
le = LabelEncoder()
df['crime_encoded'] = le.fit_transform(df['primary_type'].fillna('OTHER'))

def location_group(loc):
    if pd.isna(loc): return 5
    loc = str(loc).upper()
    if any(w in loc for w in ['STREET','ALLEY','SIDEWALK','PARKING']): return 0
    if any(w in loc for w in ['RESIDENCE','APARTMENT','HOUSE']): return 1
    if any(w in loc for w in ['STORE','RESTAURANT','BANK','RETAIL','GAS']): return 2
    if any(w in loc for w in ['CTA','BUS','TRAIN','VEHICLE']): return 3
    if any(w in loc for w in ['PARK','SCHOOL','CHURCH','HOSPITAL']): return 4
    return 5

df['location_group'] = df['location_description'].apply(location_group)

print(f'\nRepartition cible :')
vc = df['arrest'].value_counts()
print(f'  Pas d\'arrestation : {vc[0]:,} ({vc[0]/len(df)*100:.1f}%)')
print(f'  Arrestation       : {vc[1]:,} ({vc[1]/len(df)*100:.1f}%)')
print(f'\nFeatures : hour, day_of_week, month, is_weekend, domestic, crime_encoded, location_group')


## 4 — Modèle Non Supervisé : K-Means Clustering

### Justification
K-Means regroupe automatiquement les crimes par **proximité géographique** sans supervision.
On identifie ainsi les **hotspots** de la ville pour comprendre la distribution spatiale.

**Features :** latitude, longitude (clustering spatial pur)
**Métrique :** Inertie (méthode Elbow pour choisir k)


In [ ]:
coords = df[['latitude','longitude']].dropna().sample(min(50000, len(df)), random_state=RANDOM_STATE)
scaler_kmeans = StandardScaler()
X_spatial = scaler_kmeans.fit_transform(coords)

# Méthode Elbow
inertias = []
K_range = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=5)
    km.fit(X_spatial)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(K_range, inertias, 'o-', color='#e74c3c', linewidth=2.5, markersize=8)
ax.axvline(x=5, color='#3498db', linestyle='--', linewidth=1.8, label='k=5 choisi')
ax.set_xlabel('Nombre de clusters (k)', fontsize=12)
ax.set_ylabel('Inertie', fontsize=12)
ax.set_title('Méthode Elbow — Choix du k optimal', fontsize=13, fontweight='bold')
ax.legend(); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('viz_04_elbow.png', bbox_inches='tight')
plt.show()
print('Coude visible à k=5 : choix retenu')


In [ ]:
K_OPTIMAL = 5
kmeans = KMeans(n_clusters=K_OPTIMAL, random_state=RANDOM_STATE, n_init=10)
kmeans.fit(X_spatial)

coords_copy = coords.copy()
coords_copy['cluster'] = kmeans.labels_
df = df.merge(coords_copy[['cluster']], left_index=True, right_index=True, how='left')
df['cluster'] = df['cluster'].fillna(0).astype(int)

print(f'K-Means : {K_OPTIMAL} clusters identifiés')
print(df['cluster'].value_counts().sort_index())


In [ ]:
# ── VIZ 4 : Clusters spatiaux + taux d'arrestation ──
CLUSTER_COLORS = ['#FF6B6B','#FFD93D','#6BCB77','#4D96FF','#C77DFF']
CLUSTER_LABELS = ['Zone A · Downtown','Zone B · North Side','Zone C · West Side',
                  'Zone D · South Side','Zone E · Far South']

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sample = df.dropna(subset=['latitude','longitude']).sample(min(15000,len(df)), random_state=42)
for c_id in range(K_OPTIMAL):
    mask = sample['cluster'] == c_id
    axes[0].scatter(sample.loc[mask,'longitude'], sample.loc[mask,'latitude'],
                    c=CLUSTER_COLORS[c_id], s=3, alpha=0.45, label=CLUSTER_LABELS[c_id])

cents_orig = scaler_kmeans.inverse_transform(kmeans.cluster_centers_)
axes[0].scatter(cents_orig[:,1], cents_orig[:,0], c='black', s=200, marker='X',
                zorder=10, label='Centroïdes')
axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')
axes[0].set_title('Clusters K-Means — Chicago', fontweight='bold')
axes[0].legend(fontsize=9, markerscale=3)

arrest_cluster = df.groupby('cluster')['arrest'].mean() * 100
axes[1].bar([CLUSTER_LABELS[i].split('·')[0].strip() for i in range(K_OPTIMAL)],
            arrest_cluster.values, color=CLUSTER_COLORS, edgecolor='white', linewidth=0.5)
axes[1].axhline(y=df['arrest'].mean()*100, color='black', linestyle='--', linewidth=1.5,
                label=f"Moy. : {df['arrest'].mean()*100:.1f}%")
axes[1].set_ylabel("Taux d'arrestation (%)")
axes[1].set_title("Taux d'Arrestation par Zone K-Means", fontweight='bold')
axes[1].legend()

plt.suptitle('Analyse Spatiale K-Means', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('viz_05_kmeans.png', bbox_inches='tight')
plt.show()


## 5 — Modèles Supervisés : Comparaison de 3 Approches

### Stratégie de comparaison
On compare **3 modèles de complexité croissante** pour justifier nos choix :

| Modèle | Famille | Complexité | Justification |
|---|---|---|---|
| Logistic Regression | Linéaire | Faible | Baseline de référence |
| Random Forest | Ensemble (Bagging) | Moyenne | Robuste, interprétable |
| XGBoost | Ensemble (Boosting) | Élevée | State-of-the-art sur données tabulaires |

**Métriques retenues :**
- **Accuracy** : proportion globale de bonnes prédictions
- **F1-Score (weighted)** : balance précision/rappel — adapté au déséquilibre des classes
- **ROC-AUC** : capacité discriminante indépendante du seuil de décision


In [ ]:
# ── Préparation Train/Test ──
FEATURES = ['latitude','longitude','hour','day_of_week','month',
            'is_weekend','domestic','cluster','location_group','crime_encoded']
TARGET = 'arrest'

df_ml = df[FEATURES + [TARGET]].dropna()
X = df_ml[FEATURES]
y = df_ml[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f'Dataset ML : {len(df_ml):,} lignes')
print(f'Train : {len(X_train):,} | Test : {len(X_test):,} (80/20 stratifié)')
print(f'Classe 0 (pas d\'arrest.) : {(y==0).mean()*100:.1f}%')
print(f'Classe 1 (arrest.)        : {(y==1).mean()*100:.1f}%')


### 5.1 — Baseline : Régression Logistique

Modèle linéaire simple. Sert de **référence** pour évaluer le gain des modèles plus complexes.

In [ ]:
print('Entrainement Logistic Regression...')

# Normalisation nécessaire pour la LR
scaler_lr = StandardScaler()
X_train_sc = scaler_lr.fit_transform(X_train)
X_test_sc  = scaler_lr.transform(X_test)

lr = LogisticRegression(
    class_weight='balanced',
    max_iter=500,
    random_state=RANDOM_STATE,
    C=1.0
)
lr.fit(X_train_sc, y_train)

y_pred_lr   = lr.predict(X_test_sc)
y_prob_lr   = lr.predict_proba(X_test_sc)[:,1]
acc_lr  = accuracy_score(y_test, y_pred_lr)
f1_lr   = f1_score(y_test, y_pred_lr, average='weighted')
auc_lr  = roc_auc_score(y_test, y_prob_lr)

print(f'\nLOGISTIC REGRESSION :')
print(f'   Accuracy  : {acc_lr*100:.2f}%')
print(f'   F1-Score  : {f1_lr*100:.2f}%')
print(f'   ROC-AUC   : {auc_lr:.4f}')
print(f'\n{classification_report(y_test, y_pred_lr, target_names=["Pas d\'arrest.","Arrestation"])}')


### 5.2 — Random Forest (Bagging)

Ensemble de 200 arbres de décision entraînés sur des sous-échantillons aléatoires. Plus robuste et capable de capturer des relations non-linéaires.

In [ ]:
print('Entrainement Random Forest...')

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight='balanced',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
rf.fit(X_train, y_train)

y_pred_rf  = rf.predict(X_test)
y_prob_rf  = rf.predict_proba(X_test)[:,1]
acc_rf  = accuracy_score(y_test, y_pred_rf)
f1_rf   = f1_score(y_test, y_pred_rf, average='weighted')
auc_rf  = roc_auc_score(y_test, y_prob_rf)

print(f'\nRANDOM FOREST :')
print(f'   Accuracy  : {acc_rf*100:.2f}%')
print(f'   F1-Score  : {f1_rf*100:.2f}%')
print(f'   ROC-AUC   : {auc_rf:.4f}')
print(f'\n{classification_report(y_test, y_pred_rf, target_names=["Pas d\'arrest.","Arrestation"])}')


### 5.3 — XGBoost (Gradient Boosting)

Construit les arbres **séquentiellement** en corrigeant les erreurs du modèle précédent. Généralement le plus performant sur les données tabulaires.

In [ ]:
print('Entrainement XGBoost...')

# Calculer scale_pos_weight pour gérer le déséquilibre
ratio = (y_train == 0).sum() / (y_train == 1).sum()

xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=ratio,
    random_state=RANDOM_STATE,
    eval_metric='logloss',
    verbosity=0,
    n_jobs=-1
)
xgb.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

y_pred_xgb = xgb.predict(X_test)
y_prob_xgb = xgb.predict_proba(X_test)[:,1]
acc_xgb = accuracy_score(y_test, y_pred_xgb)
f1_xgb  = f1_score(y_test, y_pred_xgb, average='weighted')
auc_xgb = roc_auc_score(y_test, y_prob_xgb)

print(f'\nXGBOOST :')
print(f'   Accuracy  : {acc_xgb*100:.2f}%')
print(f'   F1-Score  : {f1_xgb*100:.2f}%')
print(f'   ROC-AUC   : {auc_xgb:.4f}')
print(f'\n{classification_report(y_test, y_pred_xgb, target_names=["Pas d\'arrest.","Arrestation"])}')


## 6 — Comparaison des Modèles

In [ ]:
# ── Tableau récapitulatif ──
results = pd.DataFrame({
    'Modèle':    ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Accuracy':  [acc_lr*100, acc_rf*100, acc_xgb*100],
    'F1-Score':  [f1_lr*100,  f1_rf*100,  f1_xgb*100],
    'ROC-AUC':   [auc_lr,     auc_rf,     auc_xgb],
    'Complexité':['Faible','Moyenne','Élevée'],
    'Interprét.':['Très haute','Haute','Moyenne'],
})
print(results.to_string(index=False))


In [ ]:
# ── VIZ 5 : Comparaison visuelle des modèles ──
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

models    = ['Logistic\nRegression', 'Random\nForest', 'XGBoost']
accs      = [acc_lr*100, acc_rf*100, acc_xgb*100]
f1s       = [f1_lr*100,  f1_rf*100,  f1_xgb*100]
aucs      = [auc_lr,     auc_rf,     auc_xgb]
colors    = ['#95a5a6', '#3498db', '#e74c3c']

# Accuracy
bars = axes[0].bar(models, accs, color=colors, edgecolor='white', linewidth=0.5, width=0.5)
axes[0].set_ylim(0, 100); axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Accuracy', fontweight='bold', fontsize=13)
axes[0].axhline(y=50, color='gray', linestyle=':', linewidth=1, label='Aléatoire (50%)')
for bar, val in zip(bars, accs):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 f'{val:.1f}%', ha='center', fontweight='bold', fontsize=11)
axes[0].legend(fontsize=9)
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

# F1-Score
bars = axes[1].bar(models, f1s, color=colors, edgecolor='white', linewidth=0.5, width=0.5)
axes[1].set_ylim(0, 100); axes[1].set_ylabel('F1-Score (%)')
axes[1].set_title('F1-Score (weighted)', fontweight='bold', fontsize=13)
for bar, val in zip(bars, f1s):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
                 f'{val:.1f}%', ha='center', fontweight='bold', fontsize=11)
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

# ROC-AUC
bars = axes[2].bar(models, aucs, color=colors, edgecolor='white', linewidth=0.5, width=0.5)
axes[2].set_ylim(0, 1); axes[2].set_ylabel('ROC-AUC')
axes[2].set_title('ROC-AUC', fontweight='bold', fontsize=13)
axes[2].axhline(y=0.5, color='gray', linestyle=':', linewidth=1, label='Aléatoire (0.5)')
for bar, val in zip(bars, aucs):
    axes[2].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                 f'{val:.3f}', ha='center', fontweight='bold', fontsize=11)
axes[2].legend(fontsize=9)
axes[2].spines['top'].set_visible(False); axes[2].spines['right'].set_visible(False)

plt.suptitle('Comparaison des 3 Modèles Supervisés', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('viz_06_model_comparison.png', bbox_inches='tight')
plt.show()
print('viz_06_model_comparison.png sauvegardé')


In [ ]:
# ── VIZ 6 : Courbes ROC superposées ──
fpr_lr,  tpr_lr,  _ = roc_curve(y_test, y_prob_lr)
fpr_rf,  tpr_rf,  _ = roc_curve(y_test, y_prob_rf)
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_prob_xgb)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr_lr,  tpr_lr,  color='#95a5a6', linewidth=2,   label=f'Logistic Regression (AUC={auc_lr:.3f})')
ax.plot(fpr_rf,  tpr_rf,  color='#3498db', linewidth=2.5, label=f'Random Forest       (AUC={auc_rf:.3f})')
ax.plot(fpr_xgb, tpr_xgb, color='#e74c3c', linewidth=2.5, label=f'XGBoost             (AUC={auc_xgb:.3f})', linestyle='--')
ax.plot([0,1],[0,1], color='lightgray', linestyle=':', linewidth=1.2, label='Aléatoire (AUC=0.5)')
ax.fill_between(fpr_xgb, tpr_xgb, alpha=0.05, color='#e74c3c')
ax.set_xlabel('Taux de Faux Positifs (FPR)', fontsize=12)
ax.set_ylabel('Taux de Vrais Positifs (TPR)', fontsize=12)
ax.set_title('Courbes ROC — Comparaison des 3 Modèles', fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('viz_07_roc_curves.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── VIZ 7 : Feature Importances RF + XGBoost côte à côte ──
fi_rf  = pd.Series(rf.feature_importances_,  index=FEATURES).sort_values()
fi_xgb = pd.Series(xgb.feature_importances_, index=FEATURES).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
col_rf  = ['#2ecc71' if v > fi_rf.median()  else '#bdc3c7' for v in fi_rf.values]
col_xgb = ['#e74c3c' if v > fi_xgb.median() else '#bdc3c7' for v in fi_xgb.values]

axes[0].barh(fi_rf.index,  fi_rf.values*100,  color=col_rf,  edgecolor='white')
axes[0].set_title('Feature Importances — Random Forest', fontweight='bold')
axes[0].set_xlabel('Importance (%)')
axes[0].spines['top'].set_visible(False); axes[0].spines['right'].set_visible(False)

axes[1].barh(fi_xgb.index, fi_xgb.values*100, color=col_xgb, edgecolor='white')
axes[1].set_title('Feature Importances — XGBoost', fontweight='bold')
axes[1].set_xlabel('Importance (%)')
axes[1].spines['top'].set_visible(False); axes[1].spines['right'].set_visible(False)

plt.suptitle('Variables les Plus Importantes (RF vs XGBoost)', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('viz_08_feature_importances.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Cross-Validation du meilleur modèle ──
print('Cross-validation 5-fold sur XGBoost (meilleur modèle)...')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(xgb, X, y, cv=cv, scoring='accuracy', n_jobs=-1)

print(f'\nCross-Validation XGBoost (5-fold) :')
print(f'   Scores     : {[f"{s*100:.2f}%" for s in cv_scores]}')
print(f'   Moyenne    : {cv_scores.mean()*100:.2f}%')
print(f'   Écart-type : ±{cv_scores.std()*100:.2f}%')
print(f'\nVariance faible = modèle stable, pas de surapprentissage')


## 7 — Insights & Conclusions

Les 3 insights suivants sont calculés **dynamiquement** depuis les données réelles.


In [ ]:
print('='*65)
print('   INSIGHTS PRINCIPAUX')
print('='*65)

# ── INSIGHT 1 : Type de crime ──
arrest_by_crime = df.groupby('primary_type')['arrest'].mean().sort_values(ascending=False)
top3_high = arrest_by_crime.head(3)
top3_low  = arrest_by_crime.tail(3)

print(f'\nINSIGHT 1 — Le type de crime est le facteur #1 (feature importance RF)')
print(f'   Crimes avec le PLUS fort taux d\'arrestation :')
for crime, rate in top3_high.items():
    print(f'     {crime:<30} -> {rate*100:.1f}%')
print(f'   Crimes avec le PLUS faible taux :')
for crime, rate in top3_low.items():
    print(f'     {crime:<30} -> {rate*100:.1f}%')
ratio_gap = top3_high.iloc[0] / top3_low.iloc[-1]
print(f'   Ratio max/min : {ratio_gap:.1f}x  (énorme dispersion)')

# ── INSIGHT 2 : Heure ──
hour_arrest = df.groupby('hour')['arrest'].mean()
peak_arrest_hour = hour_arrest.idxmax()
low_arrest_hour  = hour_arrest.idxmin()
night_rate = df[df['hour'].between(0,5)]['arrest'].mean()
day_rate   = df[df['hour'].between(10,16)]['arrest'].mean()

print(f'\nINSIGHT 2 — L\'heure influence fortement le taux d\'arrestation')
print(f'   Heure avec le + d\'arrestations : {peak_arrest_hour}h ({hour_arrest[peak_arrest_hour]*100:.1f}%)')
print(f'   Heure avec le - d\'arrestations : {low_arrest_hour}h ({hour_arrest[low_arrest_hour]*100:.1f}%)')
print(f'   Nuit (0h–5h)   : {night_rate*100:.1f}% d\'arrestations')
print(f'   Journée (10h–16h) : {day_rate*100:.1f}% d\'arrestations')
print(f'   Les crimes de nuit sont moins souvent résolus (-{(day_rate-night_rate)*100:.1f}%)')

# ── INSIGHT 3 : Zone géographique ──
arrest_by_cluster = df.groupby('cluster')['arrest'].mean()
best_cluster  = arrest_by_cluster.idxmax()
worst_cluster = arrest_by_cluster.idxmin()

print(f'\nINSIGHT 3 — La zone géographique corrèle avec le type de crime dominant')
print(f'   Cluster {best_cluster} : {arrest_by_cluster[best_cluster]*100:.1f}% d\'arrestations (dominé par Narcotics)')
print(f'   Cluster {worst_cluster} : {arrest_by_cluster[worst_cluster]*100:.1f}% d\'arrestations (dominé par Theft/Fraud)')
print(f'   Les zones à forte concentration de drogue ont plus d\'arrestations')
print(f'     car les agents sont souvent en flagrant délit')

# ── RÉSUMÉ ──
best_model = max([('Logistic Regression', acc_lr, auc_lr),
                  ('Random Forest', acc_rf, auc_rf),
                  ('XGBoost', acc_xgb, auc_xgb)], key=lambda x: x[1])

print(f'\nRÉSUMÉ MODÈLES :')
print(f'   Logistic Regression : {acc_lr*100:.1f}% accuracy | AUC {auc_lr:.3f}  baseline')
print(f'   Random Forest       : {acc_rf*100:.1f}% accuracy | AUC {auc_rf:.3f}  robuste')
print(f'   XGBoost             : {acc_xgb*100:.1f}% accuracy | AUC {auc_xgb:.3f}  meilleur')
print(f'\nMeilleur modèle : {best_model[0]} ({best_model[1]*100:.1f}% accuracy)')
print(f'   Progression : +{(best_model[1]-acc_lr)*100:.1f}% vs baseline logistique')


## 8 — Sauvegarde des Modèles

In [ ]:
joblib.dump(lr,           'lr_chicago.pkl')
joblib.dump(rf,           'rf_chicago.pkl')
joblib.dump(xgb,          'xgb_chicago.pkl')
joblib.dump(kmeans,       'kmeans_chicago.pkl')
joblib.dump(scaler_kmeans,'scaler_kmeans.pkl')
joblib.dump(scaler_lr,    'scaler_lr.pkl')
joblib.dump(le,           'label_encoder.pkl')

print('Modeles sauvegardes :')
print('   lr_chicago.pkl        — Logistic Regression (baseline)')
print('   rf_chicago.pkl        — Random Forest')
print('   xgb_chicago.pkl       — XGBoost (meilleur)')
print('   kmeans_chicago.pkl    — K-Means clustering')
print()
print('Visualisations generees :')
for f in sorted(os.listdir('.')):
    if f.startswith('viz_') and f.endswith('.png'):
        size = os.path.getsize(f) // 1024
        print(f'   {f}  ({size} Ko)')


## 9- Update the data

In [1]:
# ═══════════════════════════════════════════════════
# TEST SUR DONNÉES RÉCENTES — API Chicago (7 derniers jours)
# ═══════════════════════════════════════════════════
from datetime import datetime, timedelta

# Date d'il y a 7 jours
date_limit = (datetime.now() - timedelta(days=7)).strftime('%Y-%m-%dT%H:%M:%S')

print(f' Récupération des crimes depuis {date_limit[:10]}...')

url_recent = 'https://data.cityofchicago.org/resource/ijzp-q8t2.json'
params_recent = {
    '$limit': 500,
    '$where': f"date > '{date_limit}'",
    '$order': 'date DESC'
}

r = requests.get(url_recent, params=params_recent, timeout=60)
df_recent = pd.DataFrame(r.json())
print(f' {len(df_recent)} crimes récents récupérés')
df_recent.head(3)

 Récupération des crimes depuis 2026-05-08...


NameError: name 'requests' is not defined